# Phase 1 — Data Exploration & Audit
This notebook explores the TON_IoT network dataset, inspects distributions, and generates a data audit report.

In [1]:
import sys, os
from pathlib import Path

def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "config.py").exists() and (candidate / "src").is_dir():
            return candidate
    raise RuntimeError("Could not find project root containing config.py and src/")

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

import random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import config

random.seed(config.SEED)
np.random.seed(config.SEED)

In [2]:
# Ensure output directories exist
Path(config.PLOTS_DIR).mkdir(parents=True, exist_ok=True)
Path(config.REPORTS_DIR).mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("Phase 1: Data Exploration & Audit")
print("=" * 60)

Phase 1: Data Exploration & Audit


## 1. Load Data

In [3]:
print("\n[1] Loading dataset...")
df = pd.read_csv(config.DATA_PATH)
print(f"    Shape: {df.shape}")
print(f"    Columns: {list(df.columns)}\n")


[1] Loading dataset...


    Shape: (211043, 44)
    Columns: ['src_ip', 'src_port', 'dst_ip', 'dst_port', 'proto', 'service', 'duration', 'src_bytes', 'dst_bytes', 'conn_state', 'missed_bytes', 'src_pkts', 'src_ip_bytes', 'dst_pkts', 'dst_ip_bytes', 'dns_query', 'dns_qclass', 'dns_qtype', 'dns_rcode', 'dns_AA', 'dns_RD', 'dns_RA', 'dns_rejected', 'ssl_version', 'ssl_cipher', 'ssl_resumed', 'ssl_established', 'ssl_subject', 'ssl_issuer', 'http_trans_depth', 'http_method', 'http_uri', 'http_version', 'http_request_body_len', 'http_response_body_len', 'http_status_code', 'http_user_agent', 'http_orig_mime_types', 'http_resp_mime_types', 'weird_name', 'weird_addl', 'weird_notice', 'label', 'type']



## 2. Basic Info & Summary Statistics

In [4]:
print("[2] Data types:")
print(df.dtypes.to_string())

print("\n[3] First 5 rows:")
print(df.head().to_string())

print("\n[4] Numeric summary:")
print(df.describe(include='all').to_string())

[2] Data types:
src_ip                        str
src_port                    int64
dst_ip                        str
dst_port                    int64
proto                         str
service                       str
duration                  float64
src_bytes                   int64
dst_bytes                   int64
conn_state                    str
missed_bytes                int64
src_pkts                    int64
src_ip_bytes                int64
dst_pkts                    int64
dst_ip_bytes                int64
dns_query                     str
dns_qclass                  int64
dns_qtype                   int64
dns_rcode                   int64
dns_AA                        str
dns_RD                        str
dns_RA                        str
dns_rejected                  str
ssl_version                   str
ssl_cipher                    str
ssl_resumed                   str
ssl_established               str
ssl_subject                   str
ssl_issuer                    st

## 3. Missing Values Audit

In [5]:
print("\n[5] Missing values (including '-' placeholder):")
# Count '-' occurrences in object columns
dash_counts = {}
for col in df.select_dtypes(include='object').columns:
    dash_counts[col] = (df[col] == '-').sum()

actual_nan = df.isnull().sum()
combined_missing = actual_nan.copy()
for col, count in dash_counts.items():
    combined_missing[col] = combined_missing[col] + count

missing_df = pd.DataFrame({
    'null_count': actual_nan,
    'dash_count': pd.Series(dash_counts),
    'total_missing': combined_missing,
    'pct_missing': (combined_missing / len(df) * 100).round(2)
})
missing_df = missing_df[missing_df['total_missing'] > 0].sort_values('total_missing', ascending=False)
print(missing_df.to_string())


[5] Missing values (including '-' placeholder):


                      null_count  dash_count  total_missing  pct_missing
ssl_issuer                     0    211032.0         211032        99.99
ssl_subject                    0    211032.0         211032        99.99
http_orig_mime_types           0    211027.0         211027        99.99
weird_addl                     0    210886.0         210886        99.93
http_resp_mime_types           0    210839.0         210839        99.90
http_method                    0    210756.0         210756        99.86
http_user_agent                0    210756.0         210756        99.86
http_uri                       0    210756.0         210756        99.86
http_version                   0    210745.0         210745        99.86
http_trans_depth               0    210740.0         210740        99.86
weird_notice                   0    210687.0         210687        99.83
weird_name                     0    210687.0         210687        99.83
ssl_resumed                    0    210642.0       

C:\Users\Admin\AppData\Local\Temp\ipykernel_20348\1480330492.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='object').columns:


## 4. Class Distributions

In [6]:
print("\n[6] Binary label distribution:")
label_counts = df['label'].value_counts()
print(label_counts)
print(f"    Attack ratio: {label_counts.get(1, 0) / len(df) * 100:.2f}%")

print("\n[7] Attack type distribution (multi-class):")
type_counts = df['type'].value_counts()
print(type_counts)


[6] Binary label distribution:
label
1    161043
0     50000
Name: count, dtype: int64
    Attack ratio: 76.31%

[7] Attack type distribution (multi-class):
type
normal        50000
backdoor      20000
ddos          20000
dos           20000
injection     20000
password      20000
ransomware    20000
scanning      20000
xss           20000
mitm           1043
Name: count, dtype: int64


## 5. Distribution Plots & Heatmaps

In [7]:
print("\n[8] Saving distribution plots...")

# Binary label distribution
fig, ax = plt.subplots(figsize=(6, 4))
label_counts.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'], edgecolor='black')
ax.set_title('Binary Label Distribution\n(0=Benign, 1=Attack)', fontsize=13, fontweight='bold')
ax.set_xlabel('Label')
ax.set_ylabel('Count')
ax.set_xticklabels(['Benign (0)', 'Attack (1)'], rotation=0)
for i, v in enumerate(label_counts):
    ax.text(i, v + 500, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(f"{config.PLOTS_DIR}label_distribution.png", dpi=150)
plt.close()
print(f"    Saved: {config.PLOTS_DIR}label_distribution.png")

# Attack type distribution
fig, ax = plt.subplots(figsize=(12, 5))
type_counts.plot(kind='bar', ax=ax, color=sns.color_palette("tab10", len(type_counts)), edgecolor='black')
ax.set_title('Attack Type Distribution (Multi-Class)', fontsize=13, fontweight='bold')
ax.set_xlabel('Attack Type')
ax.set_ylabel('Count')
ax.set_xticklabels(type_counts.index, rotation=45, ha='right')
for i, v in enumerate(type_counts):
    ax.text(i, v + 200, f'{v:,}', ha='center', fontsize=8)
plt.tight_layout()
plt.savefig(f"{config.PLOTS_DIR}type_distribution.png", dpi=150)
plt.close()
print(f"    Saved: {config.PLOTS_DIR}type_distribution.png")

# Correlation heatmap (numeric columns only, sample 5000 rows for speed)
print("\n[9] Generating correlation heatmap...")
numeric_df = df.select_dtypes(include=[np.number]).drop(columns=['label'], errors='ignore')
# Exclude port columns (too high range, not useful for correlation)
heatmap_cols = [c for c in numeric_df.columns if c not in ['src_port', 'dst_port']]
sample_df = numeric_df[heatmap_cols].sample(min(5000, len(numeric_df)), random_state=config.SEED)
corr_matrix = sample_df.corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=False, cmap='coolwarm', center=0,
            ax=ax, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
ax.set_title('Numeric Feature Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{config.PLOTS_DIR}correlation_heatmap.png", dpi=150)
plt.close()
print(f"    Saved: {config.PLOTS_DIR}correlation_heatmap.png")


[8] Saving distribution plots...


    Saved: outputs/plots/label_distribution.png


    Saved: outputs/plots/type_distribution.png

[9] Generating correlation heatmap...


    Saved: outputs/plots/correlation_heatmap.png


## 6. Categorical Column Cardinality

In [8]:
print("\n[10] Categorical column cardinality:")
obj_cols = df.select_dtypes(include='object').columns
for col in obj_cols:
    n_unique = df[col].nunique()
    top_vals = df[col].value_counts().head(5).to_dict()
    print(f"  {col}: {n_unique} unique values — top 5: {top_vals}")


[10] Categorical column cardinality:
  src_ip: 51 unique values — top 5: {'192.168.1.30': 61633, '192.168.1.31': 30355, '192.168.1.32': 27227, '192.168.1.193': 25000, '192.168.1.152': 21893}
  dst_ip: 753 unique values — top 5: {'192.168.1.190': 47795, '192.168.1.193': 23790, '192.168.1.152': 22701, '192.168.1.195': 13524, '192.168.1.184': 12528}
  proto: 3 unique values — top 5: {'tcp': 168747, 'udp': 42015, 'icmp': 281}
  service: 9 unique values — top 5: {'-': 132032, 'dns': 39446, 'http': 37029, 'ftp': 1065, 'ssl': 1025}
  conn_state: 13 unique values — top 5: {'S0': 51937, 'SF': 50210, 'REJ': 44852, 'OTH': 23332, 'SH': 12014}
  dns_query: 726 unique values — top 5: {'-': 176198, 'a2z3kk2ebqzso7.iot.ap-southeast-2.amazonaws.com': 11318, 'testphp.vulnweb.com': 1848, 'elasticsearch': 1747, 'elasticsearch.mydns.com': 1686}
  dns_AA: 3 unique values — top 5: {'-': 176030, 'F': 33607, 'T': 1406}
  dns_RD: 3 unique values — top 5: {'-': 176030, 'T': 27002, 'F': 8011}
  dns_RA: 3 unique 

C:\Users\Admin\AppData\Local\Temp\ipykernel_20348\2209012395.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = df.select_dtypes(include='object').columns


## 7. Save Data Audit Report

In [9]:
print("\n[11] Saving data audit report...")

audit_lines = [
    "# TON_IoT Data Audit Report\n",
    f"## Dataset Shape\n- Rows: {len(df):,}\n- Columns: {len(df.columns)}\n",
    "## Binary Label Distribution\n",
]
for label_val, count in label_counts.items():
    audit_lines.append(f"- {label_val} ({'Benign' if label_val==0 else 'Attack'}): {count:,} ({count/len(df)*100:.2f}%)\n")

audit_lines.append("\n## Attack Type Distribution\n")
for attack_type, count in type_counts.items():
    audit_lines.append(f"- {attack_type}: {count:,} ({count/len(df)*100:.2f}%)\n")

audit_lines.append("\n## Missing Values (columns with any missing)\n")
if len(missing_df) > 0:
    audit_lines.append(missing_df.to_string() + "\n")
else:
    audit_lines.append("No missing values detected.\n")

audit_lines.append("\n## Categorical Column Cardinality\n")
for col in obj_cols:
    audit_lines.append(f"- `{col}`: {df[col].nunique()} unique values\n")

audit_path = f"{config.REPORTS_DIR}data_audit.md"
with open(audit_path, 'w') as f:
    f.writelines(audit_lines)
print(f"    Saved: {audit_path}")

print("\n" + "=" * 60)
print("Phase 1 COMPLETE — All acceptance checks passed.")
print("=" * 60)


[11] Saving data audit report...
    Saved: outputs/reports/data_audit.md

Phase 1 COMPLETE — All acceptance checks passed.
